# Advanced Stock Price Prediction

Clean end-to-end notebook using the `src/stock_prediction` package.
All model and pipeline logic lives in the package; this notebook handles orchestration and visualisation.

**Target:** 5-day log return `log(Close_{t+5} / Close_t)` — stationary and scale-invariant.

**Workflow:**
1. Download OHLCV data (parquet cache)
2. Engineer 34 returns-based features
3. Train and compare 12+ models on AAPL
4. Multi-stock LassoCV pipeline (5 tickers)
5. Trading backtest with Sharpe ratio and max drawdown

## 0 · Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.style as mstyle
mstyle.use('seaborn-v0_8-darkgrid')
%matplotlib inline

from stock_prediction.config              import STOCKS, START_DATE, END_DATE, PREDICTION_HORIZON
from stock_prediction.data.loader         import download_stocks
from stock_prediction.features.engineer   import engineer_features, prepare_xy
from stock_prediction.models.evaluate     import build_comparison_table
from stock_prediction.models.train        import train_pipeline, run_all_stocks, backtest
from stock_prediction.visualization.plots import (
    plot_model_comparison, plot_residuals, plot_coefficients,
    plot_cross_stock_comparison, plot_coef_heatmap, plot_backtest,
)

print(f'Stocks  : {list(STOCKS.keys())}')
print(f'Period  : {START_DATE} → {END_DATE}')
print(f'Horizon : {PREDICTION_HORIZON} trading days')

## 1 · Download Data

In [ ]:
stock_data = download_stocks()

for ticker, df in stock_data.items():
    print(f'{ticker:6s}  {len(df):,} trading days  '
          f'({df.index.min().date()} → {df.index.max().date()})')

## 2 · Feature Engineering (AAPL preview)

34 returns-based features — all scale-invariant, no raw price levels.
Calendar features use cyclical sine/cosine encoding.

In [ ]:
from scipy import stats

df_aapl = engineer_features(stock_data['AAPL'], 'AAPL')
X_aapl, y_aapl, feat_cols = prepare_xy(df_aapl)

print(f'Feature matrix : {X_aapl.shape}')
print(f'Target mean    : {y_aapl.mean():.6f}  (≈0 confirms stationarity)')
print(f'Target std     : {y_aapl.std():.4f}')

# Multicollinearity audit
corr_matrix = X_aapl.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(c, r, upper.loc[r, c]) for c in upper.columns
             for r in upper.index if pd.notna(upper.loc[r, c]) and upper.loc[r, c] > 0.85]
print(f'\nFeature pairs |r| > 0.85 : {len(high_corr)}')

# Top Spearman correlations with target
spearman = {col: stats.spearmanr(X_aapl[col], y_aapl).statistic for col in feat_cols}
top10 = sorted(spearman.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
print('\nTop 10 features by |Spearman ρ| with target:')
for i, (f, rho) in enumerate(top10, 1):
    print(f'  {i:2d}. {f:28s}  ρ={rho:+.4f}')

## 3 · Full Model Comparison (AAPL)

Trains linear + tree + ensemble models with a temporal 80/20 holdout.
RobustScaler fit on training data only; TimeSeriesSplit used throughout.

In [ ]:
result = train_pipeline(
    stock_data['AAPL'], 'AAPL',
    run_linear=True, run_tree=True, run_ensemble=True,
    verbose=False,
)

comp_df = build_comparison_table({
    name: (res['train_metrics'], res['test_metrics'])
    for name, res in result['all_results'].items()
})

print('=' * 90)
print('  MODEL COMPARISON  (AAPL, 5-day log return target)')
print('=' * 90)
print(comp_df.to_string())
print('  *** p<0.001  ** p<0.01  * p<0.05  (ns) not significant vs. random')

best_name = comp_df.iloc[0]['Model']
print(f'\n  Best model: {best_name}')

### 3.1 · Model Comparison Charts

In [ ]:
fig = plot_model_comparison(comp_df)
plt.show()

### 3.2 · Best Linear Model — Coefficients and Residuals

In [ ]:
best_linear_name = max(
    result['linear_results'].keys(),
    key=lambda k: result['linear_results'][k]['test_metrics'].r2
)
best_linear = result['linear_results'][best_linear_name]
print(f'Best linear model : {best_linear_name}')
print(f'  Test R² = {best_linear["test_metrics"].r2:.4f}   '
      f'RMSE = {best_linear["test_metrics"].rmse:.6f}')

if hasattr(best_linear['model'], 'coef_'):
    fig = plot_coefficients(best_linear['model'], result['feature_cols'],
                            model_name=best_linear_name)
    plt.show()

fig = plot_residuals(
    result['y_test'], best_linear['predictions'],
    model_name=best_linear_name,
)
plt.show()

## 4 · Cross-Stock Analysis (LassoCV)

In [ ]:
all_stocks = run_all_stocks(stock_data, verbose=False)

print('=' * 80)
print('CROSS-STOCK PERFORMANCE (LassoCV, 5-day log return)')
print('=' * 80)

rows = []
for ticker, res in all_stocks.items():
    tr, te = res['train_metrics'], res['test_metrics']
    rows.append({
        'Stock'   : ticker,
        'Sector'  : res['sector'],
        'Train R²': f'{tr.r2:.4f}',
        'Test R²' : f'{te.r2:.4f}',
        'Gap'     : f'{tr.r2 - te.r2:+.4f}',
        'RMSE'    : f'{te.rmse:.6f}',
        'Dir Acc' : f'{te.dir_acc:.2%} {te.sig_stars}',
        'Alpha*'  : f'{res["alpha"]:.6f}',
    })

cs_df = pd.DataFrame(rows)
print(cs_df.to_string(index=False))

avg_r2  = np.mean([r['test_metrics'].r2      for r in all_stocks.values()])
avg_dir = np.mean([r['test_metrics'].dir_acc for r in all_stocks.values()])
print(f'\nAverage Test R²  : {avg_r2:.4f}')
print(f'Average Dir Acc  : {avg_dir:.2%}')

### 4.1 · Cross-Stock Charts

In [ ]:
fig = plot_cross_stock_comparison(all_stocks)
plt.show()

fig = plot_coef_heatmap(all_stocks, min_tickers=2)
plt.show()

# Most consistently selected features
coef_matrix = pd.DataFrame({t: r['coef'] for t, r in all_stocks.items()}).T
common = (coef_matrix != 0).sum(axis=0).sort_values(ascending=False)
print('\nMost consistently selected features:')
for feat, freq in common.head(10).items():
    bar = '█' * int(freq)
    print(f'  {feat:28s}  {bar}  {int(freq)}/{len(all_stocks)}')

## 5 · Trading Backtest

In [ ]:
best_linear_pred = result['linear_results'][best_linear_name]['predictions']

fig = plot_backtest(
    result['y_test'].values, best_linear_pred,
    model_name=best_linear_name,
)
plt.show()

bt = backtest(result['y_test'].values, best_linear_pred)
print(f'Sharpe ratio      : {bt["Sharpe"]:.3f}')
print(f'Annualised return : {bt["Ann_Return"]:.2%}')
print(f'Max drawdown      : {bt["Max_DD"]:.2%}')
print(f'Calmar ratio      : {bt["Calmar"]:.3f}')
print(f'Strategy total    : {bt["Total_Ret"]:.2%}')
print(f'Buy-and-hold      : {bt["BAH_Ret"]:.2%}')

# Cross-stock backtest
print('\nCross-stock backtest (LassoCV):')
bt_rows = []
for ticker, res in all_stocks.items():
    b = backtest(res['y_test'].values, res['predictions'])
    bt_rows.append({
        'Stock'  : ticker,
        'Sector' : res['sector'],
        'Sharpe' : f'{b["Sharpe"]:.3f}',
        'Ann Ret': f'{b["Ann_Return"]:.2%}',
        'Max DD' : f'{b["Max_DD"]:.2%}',
        'BaH Ret': f'{b["BAH_Ret"]:.2%}',
    })
print(pd.DataFrame(bt_rows).to_string(index=False))

## 6 · Summary

| Aspect | Details |
|--------|---------|
| **Stocks** | AAPL, MSFT, JPM, JNJ, XOM |
| **Period** | 2008–2024 (~16 years) |
| **Target** | 5-day log return (stationary) |
| **Features** | 34, returns-based |
| **Scaler** | RobustScaler |
| **CV** | TimeSeriesSplit (5 folds) |
| **Regularisation** | Auto-tuned via LassoCV / RidgeCV |
| **Ensembles** | Voting, Stacking, Blending, CV-Weighted |
| **Backtest** | Long/short, Sharpe + max drawdown |

**Honest caveats:**
- Directional accuracy ~52–56% — barely above random.
- Models capture autocorrelation and volatility regimes, not alpha-generating signals.
- No news, earnings, macro, or sentiment data incorporated.
- Position sizing matters far more than directional timing at this accuracy level.